In [0]:
from pyspark.sql.functions import col, coalesce, row_number, lit
from pyspark.sql.window import Window

# 1. Extract Distinct Locations from Silver Members
df_members = spark.read.table("hdfc_ergo.hdfc_silver.dim_members")
df_distinct_geo = df_members.select("state", "city", "pincode") \
    .dropna(subset=["state", "city"]) \
    .dropDuplicates()

# 2. Create a Reference Dataset for typo correction
# In a real project, you would read this from a seed CSV file in your volume.
# For practice, we create it here.
typo_data = [
    ("BENGLORE", "BENGALURU"),
    ("BANGALORE", "BENGALURU"),
    ("BOMBAY", "MUMBAI"),
    ("CALCUTTA", "KOLKATA"),
    ("MADRAS", "CHENNAI")
]
df_ref = spark.createDataFrame(typo_data, ["raw_city", "correct_city"])

# 3. Fix the Typos
# Left join to the reference table. If a match is found, use correct_city, else use original city.
df_fixed = df_distinct_geo.join(df_ref, df_distinct_geo.city == df_ref.raw_city, "left")
df_fixed = df_fixed.withColumn("city_standard", coalesce(col("correct_city"), col("city")))

# 4. Generate the Surrogate Key (geography_sk)
# We use a window function ordered by state and city to get a nice 1, 2, 3 sequence
windowSpec = Window.orderBy("state", "city_standard")
df_geo = df_fixed.withColumn("geography_sk", row_number().over(windowSpec))

# 5. Select Final Columns & Rename city_standard to city
df_final = df_geo.select(
    "geography_sk",
    "state",
    col("city_standard").alias("city"),
    "pincode"
)

# 6. Write to Gold Schema
df_final.write.mode("overwrite").saveAsTable("hdfc_ergo.hdfc_gold.dim_geography")

print("✅ Gold dim_geography created successfully!")
display(spark.table("hdfc_ergo.hdfc_gold.dim_geography"))